# Residuals, Intervals And Risk

Tests whether forecast errors are centered, stable across scale and adequately covered by empirical prediction intervals.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = Path("Ai miroservices/modeling/project_operational_baseline").resolve()
OUT = ROOT / "outputs"
sns.set_theme(style="whitegrid")

In [ ]:
rows=pd.read_csv(OUT/'champion_test_backtest_rows.csv.gz',parse_dates=['origin_month'])
fig,axes=plt.subplots(2,2,figsize=(14,10))
sns.scatterplot(data=rows,x='prediction',y='residual',hue='material_type',alpha=.45,ax=axes[0,0]); axes[0,0].axhline(0,color='black')
sns.histplot(rows.residual,kde=True,ax=axes[0,1])
sns.lineplot(data=rows.groupby('origin_month').residual.mean().reset_index(),x='origin_month',y='residual',marker='o',ax=axes[1,0]); axes[1,0].axhline(0,color='black')
sns.scatterplot(data=rows,x='prediction',y='absolute_error',hue='amalgamated_class',alpha=.4,legend=False,ax=axes[1,1]); plt.tight_layout(); plt.show()

In [ ]:
coverage=rows.groupby(['material_type','amalgamated_class']).interval_covered.agg(['mean','size']).reset_index()
display(coverage.sort_values('mean'))
print('Aggregate empirical coverage:',rows.interval_covered.mean())

In [ ]:
from scipy import stats
print('Jarque-Bera:',stats.jarque_bera(rows.residual))
print('Spearman |residual| vs fitted:',stats.spearmanr(rows.absolute_error,rows.prediction))